In [2]:
%pip install mne mne-icalabel meegkit

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import mne
from mne_icalabel import label_components
import numpy as np
# A biblioteca meegkit contém o port do ASR para Python
from meegkit.asr import ASR 

## Pré-processamento

In [4]:
# 1. CARREGAMENTO DOS DADOS (BIDS FORMAT)
file_path = '88 subjects/sub-001/eeg/sub-001_task-eyesclosed_eeg.set'
raw = mne.io.read_raw_eeglab(file_path, preload=True)

# 2. FILTRAGEM
raw.filter(
    l_freq=0.5, 
    h_freq=45.0, 
    method='iir', 
    iir_params=dict(order=4, ftype='butter')
)

# 3. RE-REFERENCIAMENTO
print("Canais disponíveis:", raw.ch_names)

# Reatribuindo o objeto 'raw' com os canais de referência adicionados
# (caso eles tenham sido cortados do arquivo bruto e atuem como referência implícita)
if 'A1' not in raw.ch_names and 'A2' not in raw.ch_names:
    raw = mne.add_reference_channels(raw, ref_channels=['A1', 'A2'])

raw.set_eeg_reference(ref_channels=['A1', 'A2'])

raw.drop_channels(['A1', 'A2'])

# 3.5. APLICAÇÃO DA MONTAGEM (COORDENADAS DOS ELETRODOS)
montage = mne.channels.make_standard_montage('standard_1020')

# Aplica a montagem aos dados brutos. 
# match_case=False ajuda caso haja diferenças de maiúsculas/minúsculas (ex: FZ vs Fz)
raw.set_montage(montage, match_case=False)

# 4. ASR (Artifact Subspace Reconstruction) 
# A rotina ASR remove períodos de dados ruins que excedem o desvio padrão de 17 em uma janela de 0.5 s

# 1. Extraímos a frequência de amostragem, que neste dataset é de 500 Hz
sfreq = raw.info['sfreq'] 

# 2. Inicializa o ASR. É fundamental passar o parâmetro 'sfreq' para que 
# a janela temporal seja calculada corretamente.
asr = ASR(method='euclid', cutoff=17, sfreq=sfreq) 

# 3. Extrai os dados sem usar o '.T'. O meegkit espera exatamente o padrão
# nativo do MNE, que é (n_canais, n_amostras).
raw_data = raw.get_data() 

# 4. Treina e aplica o ASR
_, sample_mask = asr.fit(raw_data)
clean_data = asr.transform(raw_data)

# 5. Devolve os dados limpos ao objeto raw do MNE
raw._data = clean_data

# 5. ANÁLISE DE COMPONENTES INDEPENDENTES (ICA)
# O algoritmo RunICA foi usado para converter os 19 sinais de EEG em 19 componentes ICA
# No MNE, o algoritmo 'RunICA' original (do EEGLAB) é acessado através do método 'infomax' estendido.
ica = mne.preprocessing.ICA(
    n_components=19, 
    method='infomax', 
    fit_params=dict(extended=True), 
    random_state=42
)
ica.fit(raw)


# 6. EXCLUSÃO AUTOMÁTICA COM ICLABEL
# Componentes ICA classificados como "eye artifacts" (artefatos oculares) ou "jaw artifacts" (artefatos de mandíbula/musculares) pelo ICLabel foram excluídos automaticamente[cite: 1].

# Extrai as classificações usando mne-icalabel
ic_labels = label_components(raw, ica, method='iclabel')
labels = ic_labels['labels']

# Mapeia os índices dos componentes que foram classificados como 'eye' ou 'muscle' (jaw)
exclude_idx = [i for i, label in enumerate(labels) if label in ['eye blink', 'muscle artifact']]
ica.exclude = exclude_idx

# Aplica a exclusão ao sinal
raw_cleaned = ica.apply(raw.copy())

# O sinal processado "raw_cleaned" está pronto para a extração de features

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 45 Hz

IIR filter parameters
---------------------
Butterworth bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 16 (effective, after forward-backward)
- Cutoffs at 0.50, 45.00 Hz: -6.02, -6.02 dB

Canais disponíveis: ['Fp1', 'Fp2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2', 'F7', 'F8', 'T3', 'T4', 'T5', 'T6', 'Fz', 'Cz', 'Pz']
EEG channel type selected for re-referencing
Applying a custom ('EEG',) reference.


C:\Users\isabe\AppData\Local\Temp\ipykernel_33452\483998837.py:19: RuntimeWarning: Location for this channel is unknown or ambiguous; consider calling set_montage() after adding new reference channels if needed. Applying a montage will only set locations of channels that exist at the time it is applied.
  raw = mne.add_reference_channels(raw, ref_channels=['A1', 'A2'])


Fitting ICA to data using 19 channels (please be patient, this may take a while)
Selecting by number: 19 components
Computing Extended Infomax ICA
Fitting ICA took 54.3s.


C:\Users\isabe\AppData\Local\Temp\ipykernel_33452\483998837.py:69: RuntimeWarning: The provided Raw instance is not filtered between 1 and 100 Hz. ICLabel was designed to classify features extracted from an EEG dataset bandpass filtered between 1 and 100 Hz (see the 'filter()' method for Raw and Epochs instances).
  ic_labels = label_components(raw, ica, method='iclabel')


Applying ICA to Raw instance
    Transforming to ICA space (19 components)
    Zeroing out 3 ICA components
    Projecting back using 19 PCA components


In [5]:
%pip install antropy tensorpac scikit-learn scipy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Features

In [6]:
# Importando as bibliotecas necessárias para a extração de features
import pandas as pd
import scipy.signal as signal
from scipy.signal import hilbert
import antropy as ant
from tensorpac import Pac
from sklearn.metrics import mutual_info_score
import os
import warnings
import logging

# Silencia todos os INFOs e WARNINGs do tensorpac, mostrando apenas ERROS
logging.getLogger('tensorpac').setLevel(logging.ERROR)

In [7]:
# Extraindo os dados do objeto MNE gerado no pré-processamento (raw_cleaned)
data = raw_cleaned.get_data()
sfreq = raw_cleaned.info['sfreq']
ch_names = raw_cleaned.ch_names

print(f"Dados extraídos: {data.shape[0]} canais e {data.shape[1]} amostras. Frequência: {sfreq}Hz.")

Dados extraídos: 19 canais e 299900 amostras. Frequência: 500.0Hz.


### Biomarcador 1 - EEG Slowing: Frequência Média Global (Overall Mean Frequency)

Captura o fenômeno de lentificação da atividade cerebral de forma unificada calculando o centro de massa do Espectro de Potência.

In [8]:
def compute_mean_frequency_per_band(data, sfreq):
    """
    Calcula a Frequência Média para cada uma das 5 bandas do EEG usando o método de Welch.
    Retorna um dicionário com 5 features.
    """
    freqs, psd = signal.welch(data, fs=sfreq, nperseg=int(2*sfreq))
    
    # Definição das 5 bandas padrão
    bands = {
        'Delta_Mean_Freq': (0.5, 4.0),
        'Theta_Mean_Freq': (4.0, 8.0),
        'Alpha_Mean_Freq': (8.0, 13.0),
        'Beta_Mean_Freq': (13.0, 25.0),
        'Gamma_Mean_Freq': (25.0, 45.0)
    }
    
    mean_freq_features = {}
    
    for band_name, (fmin, fmax) in bands.items():
        # Filtra frequências e potências apenas para a banda atual
        idx_band = np.logical_and(freqs >= fmin, freqs <= fmax)
        freqs_band = freqs[idx_band]
        psd_band = psd[:, idx_band]
        
        # Soma da potência na banda para o cálculo do peso
        sum_psd = np.sum(psd_band, axis=1)
        
        # Prevenção matemática: evita divisão por zero se a potência for absolutamente nula
        sum_psd[sum_psd == 0] = 1e-10 
        
        # Calcula o centro de massa (frequência média ponderada) para cada canal
        mean_freqs_channels = np.sum(freqs_band * psd_band, axis=1) / sum_psd
        
        # Tira a média desse valor entre todos os canais e salva no dicionário
        mean_freq_features[band_name] = np.mean(mean_freqs_channels)
        
    return mean_freq_features

### Biomarcador 2 - Complexidade do EEG: Complexidade de Lempel-Ziv (LZC)

Mede a taxa de novos padrões distintos no sinal temporal. Uma menor complexidade indica perda de variabilidade funcional associada à patologia.

In [9]:
def compute_lzc_per_band(data, sfreq):
    """
    Calcula a Lempel-Ziv Complexity para cada uma das 5 bandas do EEG.
    O sinal é filtrado, binarizado pela mediana e a complexidade é extraída.
    """
    bands = {
        'Delta_LZC': (0.5, 4.0),
        'Theta_LZC': (4.0, 8.0),
        'Alpha_LZC': (8.0, 13.0),
        'Beta_LZC': (13.0, 25.0),
        'Gamma_LZC': (25.0, 45.0)
    }
    
    lzc_features = {}
    
    for band_name, (fmin, fmax) in bands.items():
        # 1. Cria o filtro passa-banda (Butterworth ordem 4)
        nyq = 0.5 * sfreq
        low = fmin / nyq
        high = fmax / nyq
        b, a = signal.butter(4, [low, high], btype='band')
        
        # 2. Aplica o filtro no sinal original (em todos os canais)
        filtered_data = signal.filtfilt(b, a, data, axis=1)
        
        lzc_values = []
        for ch_data in filtered_data:
            # 3. Binariza o sinal filtrado: 1 se maior que a mediana, 0 caso contrário
            binarized = np.where(ch_data > np.median(ch_data), 1, 0)
            
            # 4. Calcula a complexidade LZC
            lzc = ant.lziv_complexity(binarized, normalize=True)
            lzc_values.append(lzc)
            
        # Tira a média da complexidade entre todos os canais para esta banda
        lzc_features[band_name] = np.mean(lzc_values)
        
    return lzc_features

### Biomarcador 3 - Conectividade Cerebral: Acoplamento Fase-Amplitude (PAC)

Evidencia como a fase de oscilações lentas governa a amplitude de oscilações rápidas, medindo a quebra de sincronia entre regiões corticais.

In [10]:
def compute_pac_per_pair(data, sfreq):
    """
    Calcula o Índice de Modulação (PAC) para múltiplos pares de frequências.
    Cruza frequências lentas (fase) modulando frequências rápidas (amplitude).
    Retorna um dicionário com 6 features de conectividade.
    """
    # Definição das bandas de Fase (Ondas lentas)
    phase_bands = {
        'Delta': [0.5, 4.0],
        'Theta': [4.0, 8.0],
        'Alpha': [8.0, 13.0]
    }
    
    # Definição das bandas de Amplitude (Ondas rápidas)
    amp_bands = {
        'Beta': [13.0, 25.0],
        'Gamma': [25.0, 45.0]
    }
    
    pac_features = {}
    
    # Ignora temporariamente os avisos matemáticos de divisão por zero/logaritmo 
    # gerados pelo método de entropia de Kullback-Leibler em épocas curtas
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        
        # Faz o cruzamento de todas as Fases com todas as Amplitudes
        for p_name, f_pha in phase_bands.items():
            for a_name, f_amp in amp_bands.items():
                
                feature_name = f"PAC_{p_name}_{a_name}"
                
                # idpac=(2, 0, 0) utiliza o Método de Kullback-Leibler (Tort et al., 2010)
                p = Pac(idpac=(2, 0, 0), f_pha=f_pha, f_amp=f_amp)
                
                # Extrai a matriz de PAC para o par atual
                pac_matrix = p.filterfit(sfreq, data)
                
                # Calcula a média da época ignorando eventuais falhas (NaNs) isoladas
                pac_features[feature_name] = np.nanmean(pac_matrix)
                
    return pac_features

### Biomarcador 4 - Oscilações em Bandas de Frequência: Ordem de Kuramoto

Fornece uma medida global da coerência de fase da população neuronal. Valores mais próximos de 1 indicam alta sincronização geral.

In [11]:
def compute_kuramoto_per_band(data, sfreq):
    """
    Calcula a Ordem de Kuramoto para cada uma das 5 bandas de frequência do EEG.
    Mede a sincronia de fase global da rede cerebral em ritmos específicos.
    Retorna um dicionário com 5 features de sincronização.
    """
    # Definição das 5 bandas padrão
    bands = {
        'Kuramoto_Delta': (0.5, 4.0),
        'Kuramoto_Theta': (4.0, 8.0),
        'Kuramoto_Alpha': (8.0, 13.0),
        'Kuramoto_Beta': (13.0, 25.0),
        'Kuramoto_Gamma': (25.0, 45.0)
    }
    
    kuramoto_features = {}
    
    for band_name, (fmin, fmax) in bands.items():
        # Filtro passa-banda (Butterworth ordem 4) para garantir um sinal narrow-band
        nyq = 0.5 * sfreq
        low = fmin / nyq
        high = fmax / nyq
        b, a = signal.butter(4, [low, high], btype='band')
        
        filtered_data = signal.filtfilt(b, a, data, axis=1)
        
        # Sinal analítico e fase instantânea via Transformada de Hilbert
        analytic_signal = hilbert(filtered_data, axis=1)
        instantaneous_phases = np.angle(analytic_signal)
        
        # Parâmetro R de Kuramoto
        # np.exp(1j * fases) cria os fasores no plano complexo
        complex_phasors = np.exp(1j * instantaneous_phases)
        
        # Média dos fasores entre todos os canais (axis=0) para cada instante de tempo
        mean_phasor_time = np.abs(np.mean(complex_phasors, axis=0))
        
        # Tira a média ao longo de toda a época (4 segundos)
        kuramoto_features[band_name] = np.mean(mean_phasor_time)
        
    return kuramoto_features

### Biomarcador 5 - Desorganização Funcional: Informação Mútua (MI)

Avalia a quantidade de informação compartilhada de forma não-linear entre diferentes áreas do córtex, evidenciando falhas na segregação funcional.

In [12]:
import numpy as np
import scipy.signal as signal
from sklearn.metrics import mutual_info_score

def compute_mutual_information_per_band(data, sfreq):
    """
    Calcula a Média da Informação Mútua (MI) para cada uma das 5 bandas de frequência do EEG.
    Mede o compartilhamento de informações (conectividade não-linear) entre os canais.
    Retorna um dicionário com 5 features de conectividade.
    """
    bands = {
        'MI_Delta': (0.5, 4.0),
        'MI_Theta': (4.0, 8.0),
        'MI_Alpha': (8.0, 13.0),
        'MI_Beta': (13.0, 25.0),
        'MI_Gamma': (25.0, 45.0)
    }
    
    n_channels = data.shape[0]
    
    # Função interna para discretização dos sinais usando histogramas (10 bins)
    def discretize(sig, bins=10):
        return np.digitize(sig, np.histogram_bin_edges(sig, bins=bins))
    
    mi_features = {}
    
    for band_name, (fmin, fmax) in bands.items():
        # 1. Filtro passa-banda para isolar o ritmo cerebral
        nyq = 0.5 * sfreq
        low = fmin / nyq
        high = fmax / nyq
        b, a = signal.butter(4, [low, high], btype='band')
        
        filtered_data = signal.filtfilt(b, a, data, axis=1)
        
        # 2. Discretiza os dados já filtrados
        discrete_data = np.apply_along_axis(discretize, 1, filtered_data)
        
        mi_values = []
        
        # 3. Loop combinatório para pareamento e cálculo entre todos os canais
        for i in range(n_channels):
            for j in range(i + 1, n_channels):
                mi = mutual_info_score(discrete_data[i], discrete_data[j])
                mi_values.append(mi)
                
        # 4. Salva a média da Informação Mútua global da rede para esta banda
        mi_features[band_name] = np.mean(mi_values)
        
    return mi_features

### Execução e Visualização das Features

In [13]:
# Usando o operador ** para desempacotar e fundir todos os dicionários resultantes
features_dict = {
    **compute_mean_frequency_per_band(data, sfreq),
    **compute_lzc_per_band(data, sfreq),
    **compute_pac_per_pair(data, sfreq),
    **compute_kuramoto_per_band(data, sfreq),
    **compute_mutual_information_per_band(data, sfreq)
}

# Convertendo para Pandas DataFrame para visualização clara no Notebook
df_features = pd.DataFrame([features_dict])

# Adicionando identificação (Útil para quando for processar vários sujeitos)
df_features.insert(0, 'Subject_ID', 'sub-001')

# Exibindo a tabela gerada
print(f"Total de features extraídas: {df_features.shape[1] - 1}") # -1 para descontar o Subject_ID
display(df_features)

Total de features extraídas: 26


,Subject_ID,Delta_Mean_Freq,Theta_Mean_Freq,Alpha_Mean_Freq,Beta_Mean_Freq,Gamma_Mean_Freq,Delta_LZC,Theta_LZC,Alpha_LZC,Beta_LZC,...,Kuramoto_Delta,Kuramoto_Theta,Kuramoto_Alpha,Kuramoto_Beta,Kuramoto_Gamma,MI_Delta,MI_Theta,MI_Alpha,MI_Beta,MI_Gamma
0,sub-001,1.296406,5.350387,9.946334,17.634239,31.744906,0.04938,0.123854,0.15975,0.259666,...,0.974432,0.860281,0.855114,0.83488,0.700491,1.07079,0.562311,0.508712,0.424167,0.16758


#### Fazendo para os 65 sujeitos (Alzheimer e Controles Saudáveis)

In [14]:
# Lista para armazenar os dicionários de features de cada sujeito
all_subjects_features = []

# O dataset contém 88 sujeitos no total (rodando para os 65 validados)
for i in range(1, 66):
    # Formata o número com zeros à esquerda (ex: 001, 002... 065)
    sub_id = f"sub-{i:03d}"
    
    # Monta o caminho do arquivo respeitando a estrutura BIDS
    file_path = f"88 subjects/{sub_id}/eeg/{sub_id}_task-eyesclosed_eeg.set"
    
    # Checagem de segurança caso algum arquivo esteja faltando
    if not os.path.exists(file_path):
        print(f"[{sub_id}] Arquivo não encontrado. Pulando...")
        continue
        
    print(f"\n{'='*50}\nIniciando processamento: {sub_id}\n{'='*50}")
    
    try:

        # PRÉ-PROCESSAMENTO

        raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose='ERROR')
        
        # 1. Filtro
        raw.filter(l_freq=0.5, h_freq=45.0, method='iir', 
                   iir_params=dict(order=4, ftype='butter'), verbose='ERROR')
        
        # 2. Re-referenciamento e Montagem
        if 'A1' not in raw.ch_names and 'A2' not in raw.ch_names:
            raw = mne.add_reference_channels(raw, ref_channels=['A1', 'A2'])
        raw.set_eeg_reference(ref_channels=['A1', 'A2'], verbose='ERROR')
        raw.drop_channels(['A1', 'A2'])
        montage = mne.channels.make_standard_montage('standard_1020')
        raw.set_montage(montage, match_case=False)
        
        # 3. ASR
        sfreq = raw.info['sfreq'] 
        asr = ASR(method='euclid', cutoff=17, sfreq=sfreq)
        raw_data = raw.get_data() 
        _, sample_mask = asr.fit(raw_data)
        raw._data = asr.transform(raw_data)
        
        # 4. ICA e ICLabel
        ica = mne.preprocessing.ICA(n_components=19, method='infomax', 
                                    fit_params=dict(extended=True), random_state=42)
        ica.fit(raw, verbose='ERROR')
        ic_labels = label_components(raw, ica, method='iclabel')
        
        # Rótulos ajustados para a nomenclatura do Python
        exclude_idx = [idx for idx, label in enumerate(ic_labels['labels']) 
                       if label in ['eye blink', 'muscle artifact']]
        ica.exclude = exclude_idx
        raw_cleaned = ica.apply(raw.copy(), verbose='ERROR')

        # EPOCHING (4s com 50% de overlap)

        # Cria as épocas de 4 segundos com 2 segundos de overlap (50%)
        epochs = mne.make_fixed_length_epochs(raw_cleaned, duration=4.0, overlap=2.0, preload=True, verbose='ERROR')
        
        # Extrai os dados no formato (n_epochs, n_channels, n_times)
        epochs_data = epochs.get_data()
        ch_names = epochs.ch_names
        
        print(f"[{sub_id}] Extraindo biomarcadores multidimensionais de {len(epochs_data)} épocas...")
        
        # EXTRAÇÃO DE FEATURES POR ÉPOCA

        for epoch_idx, epoch_data in enumerate(epochs_data):
            
            # Extrai as features apenas para a janela temporal de 4 segundos atual
            # utilizando o desempacotamento de dicionário (**) para expandir as bandas
            features_dict = {
                "Subject_ID": sub_id,
                "Epoch_ID": f"{sub_id}_ep{epoch_idx:04d}", 
                **compute_mean_frequency_per_band(epoch_data, sfreq),
                **compute_lzc_per_band(epoch_data, sfreq),
                **compute_pac_per_pair(epoch_data, sfreq),
                **compute_kuramoto_per_band(epoch_data, sfreq),
                **compute_mutual_information_per_band(epoch_data, sfreq)
            }
            
            all_subjects_features.append(features_dict)
            
        print(f"[{sub_id}] Finalizado com sucesso!")
        
    except Exception as e:
        print(f"[{sub_id}] Erro durante o processamento: {e}")

# CONSOLIDAÇÃO FINAL

# Cria o DataFrame consolidado com as features dos 65 indivíduos
df_all_features = pd.DataFrame(all_subjects_features)

# Salva em um arquivo CSV para treinamento do modelo de Machine Learning
df_all_features.to_csv('eeg_features.csv', index=False)

print("\nProcessamento em lote concluído! Dados salvos em 'eeg_features_dataset_multivariado.csv'.")
display(df_all_features.head())


Iniciando processamento: sub-001


C:\Users\isabe\AppData\Local\Temp\ipykernel_33452\1466293135.py:31: RuntimeWarning: Location for this channel is unknown or ambiguous; consider calling set_montage() after adding new reference channels if needed. Applying a montage will only set locations of channels that exist at the time it is applied.
  raw = mne.add_reference_channels(raw, ref_channels=['A1', 'A2'])
C:\Users\isabe\AppData\Local\Temp\ipykernel_33452\1466293135.py:48: RuntimeWarning: The provided Raw instance is not filtered between 1 and 100 Hz. ICLabel was designed to classify features extracted from an EEG dataset bandpass filtered between 1 and 100 Hz (see the 'filter()' method for Raw and Epochs instances).
  ic_labels = label_components(raw, ica, method='iclabel')


[sub-001] Extraindo biomarcadores multidimensionais de 298 épocas...
[sub-001] Finalizado com sucesso!

Iniciando processamento: sub-002


C:\Users\isabe\AppData\Local\Temp\ipykernel_33452\1466293135.py:31: RuntimeWarning: Location for this channel is unknown or ambiguous; consider calling set_montage() after adding new reference channels if needed. Applying a montage will only set locations of channels that exist at the time it is applied.
  raw = mne.add_reference_channels(raw, ref_channels=['A1', 'A2'])
C:\Users\isabe\AppData\Local\Temp\ipykernel_33452\1466293135.py:48: RuntimeWarning: The provided Raw instance is not filtered between 1 and 100 Hz. ICLabel was designed to classify features extracted from an EEG dataset bandpass filtered between 1 and 100 Hz (see the 'filter()' method for Raw and Epochs instances).
  ic_labels = label_components(raw, ica, method='iclabel')


[sub-002] Extraindo biomarcadores multidimensionais de 395 épocas...
[sub-002] Finalizado com sucesso!

Iniciando processamento: sub-003


...

C:\Users\isabe\AppData\Local\Temp\ipykernel_33452\1466293135.py:31: RuntimeWarning: Location for this channel is unknown or ambiguous; consider calling set_montage() after adding new reference channels if needed. Applying a montage will only set locations of channels that exist at the time it is applied.
  raw = mne.add_reference_channels(raw, ref_channels=['A1', 'A2'])
C:\Users\isabe\AppData\Local\Temp\ipykernel_33452\1466293135.py:48: RuntimeWarning: The provided Raw instance is not filtered between 1 and 100 Hz. ICLabel was designed to classify features extracted from an EEG dataset bandpass filtered between 1 and 100 Hz (see the 'filter()' method for Raw and Epochs instances).
  ic_labels = label_components(raw, ica, method='iclabel')


[sub-064] Extraindo biomarcadores multidimensionais de 423 épocas...
[sub-064] Finalizado com sucesso!

Iniciando processamento: sub-065


C:\Users\isabe\AppData\Local\Temp\ipykernel_33452\1466293135.py:31: RuntimeWarning: Location for this channel is unknown or ambiguous; consider calling set_montage() after adding new reference channels if needed. Applying a montage will only set locations of channels that exist at the time it is applied.
  raw = mne.add_reference_channels(raw, ref_channels=['A1', 'A2'])
C:\Users\isabe\AppData\Local\Temp\ipykernel_33452\1466293135.py:48: RuntimeWarning: The provided Raw instance is not filtered between 1 and 100 Hz. ICLabel was designed to classify features extracted from an EEG dataset bandpass filtered between 1 and 100 Hz (see the 'filter()' method for Raw and Epochs instances).
  ic_labels = label_components(raw, ica, method='iclabel')


[sub-065] Extraindo biomarcadores multidimensionais de 441 épocas...
[sub-065] Finalizado com sucesso!

Processamento em lote concluído! Dados salvos em 'eeg_features_dataset_multivariado.csv'.


,Subject_ID,Epoch_ID,Delta_Mean_Freq,Theta_Mean_Freq,Alpha_Mean_Freq,Beta_Mean_Freq,Gamma_Mean_Freq,Delta_LZC,Theta_LZC,Alpha_LZC,...,Kuramoto_Delta,Kuramoto_Theta,Kuramoto_Alpha,Kuramoto_Beta,Kuramoto_Gamma,MI_Delta,MI_Theta,MI_Alpha,MI_Beta,MI_Gamma
0,sub-001,sub-001_ep0000,1.640240,5.556964,9.578184,17.539659,32.905927,0.064929,0.157850,0.197673,...,0.952309,0.886667,0.889152,0.853127,0.683875,1.256927,0.782428,0.719417,0.557453,0.269773
1,sub-001,sub-001_ep0001,1.263603,5.297219,9.978289,17.177341,32.587770,0.069258,0.145152,0.201713,...,0.972943,0.832986,0.855320,0.862718,0.680270,1.235947,0.655105,0.639406,0.589453,0.271156
2,sub-001,sub-001_ep0002,1.241174,5.504575,10.279413,17.791362,32.251291,0.065506,0.138227,0.210081,...,0.961834,0.855898,0.864054,0.849082,0.665067,1.210207,0.726269,0.618728,0.592048,0.237782
3,sub-001,sub-001_ep0003,1.896750,5.383784,10.495496,17.017366,31.464986,0.060889,0.145441,0.193056,...,0.962289,0.955732,0.924691,0.861565,0.647490,1.268228,0.839995,0.756799,0.673875,0.236825
4,sub-001,sub-001_ep0004,1.362652,5.152837,10.775765,17.312952,31.387263,0.061755,0.153232,0.196518,...,0.968564,0.877555,0.886577,0.870110,0.627593,1.384782,0.683039,0.752469,0.672838,0.223810


In [16]:
df_model_ready = pd.read_csv('eeg_features.csv')

# 1. Carrega o arquivo com os diagnósticos fornecido pelo dataset original
df_labels = pd.read_csv('88 subjects/participants.tsv', sep='\t')

# 2. Padroniza o nome da coluna de ID usando 'participant_id'
df_labels = df_labels.rename(columns={'participant_id': 'Subject_ID'})

# 3. Junta as features limpas com a coluna 'Group' (A=AD, C=CN)
df_final = pd.merge(df_model_ready, df_labels[['Subject_ID', 'Group']], on='Subject_ID', how='left')

# 4. Salva o CSV definitivo para o treinamento
df_final.to_csv('eeg_features_for_ML.csv', index=False)

print(f"Arquivo CSV gerado com sucesso! Shape final: {df_final.shape}")

Arquivo CSV gerado com sucesso! Shape final: (26823, 29)


### Criando a baseline de comparação

In [17]:
def compute_baseline_rbp(data, sfreq):
    """
    Calcula a Potência Relativa de Banda (RBP) de 5 frequências usando o método de Welch.
    Esta é a replicação exata da metodologia do artigo.
    """
    # 1. PSD usando o método de Welch
    freqs, psd = signal.welch(data, fs=sfreq, nperseg=int(2*sfreq))
    
    # 2. Espectro total de interesse (0.5 a 45 Hz) para calcular a potência relativa
    idx_total = np.logical_and(freqs >= 0.5, freqs <= 45.0)
    total_power = np.sum(psd[:, idx_total], axis=1)
    
    # 3. Definição estrita das 5 bandas de frequência do artigo
    bands = {
        'Delta_RBP': (0.5, 4.0),
        'Theta_RBP': (4.0, 8.0),
        'Alpha_RBP': (8.0, 13.0),
        'Beta_RBP': (13.0, 25.0),
        'Gamma_RBP': (25.0, 45.0)
    }
    
    rbp_features = {}
    
    for band_name, (fmin, fmax) in bands.items():
        # Isola as frequências da banda específica
        idx_band = np.logical_and(freqs >= fmin, freqs <= fmax)
        
        # Potência absoluta da banda
        band_power = np.sum(psd[:, idx_band], axis=1)
        
        # Potência relativa = Potência da banda / Potência total (0.5-45 Hz)
        relative_power = band_power / total_power
        
        # Tira a média dos canais para ter uma feature global por época
        rbp_features[band_name] = np.mean(relative_power)
        
    return rbp_features

In [18]:
df_filtered = pd.read_csv('eeg_features_for_ML.csv')

# Pega a lista única de IDs de sujeitos para o processamento da baseline
subjects_to_process = df_filtered['Subject_ID'].unique()

print(f"Total de pacientes selecionados para a baseline: {len(subjects_to_process)} (AD e CN)")

baseline_features_list = []

Total de pacientes selecionados para a baseline: 65 (AD e CN)


In [19]:
for sub_id in subjects_to_process:
    file_path = f"88 subjects/{sub_id}/eeg/{sub_id}_task-eyesclosed_eeg.set"
    
    if not os.path.exists(file_path):
        continue
        
    print(f"[{sub_id}] Extraindo Baseline (RBP)...")
    
    try:
        # Refaz rapidamente o pré-processamento para garantir alinhamento perfeito de sinal
        raw = mne.io.read_raw_eeglab(file_path, preload=True, verbose='ERROR')
        raw.filter(l_freq=0.5, h_freq=45.0, method='iir', iir_params=dict(order=4, ftype='butter'), verbose='ERROR')
        
        if 'A1' not in raw.ch_names and 'A2' not in raw.ch_names:
            raw = mne.add_reference_channels(raw, ref_channels=['A1', 'A2'])
        raw.set_eeg_reference(ref_channels=['A1', 'A2'], verbose='ERROR')
        raw.drop_channels(['A1', 'A2'])
        raw.set_montage(mne.channels.make_standard_montage('standard_1020'), match_case=False)
        
        sfreq = raw.info['sfreq'] 
        asr = ASR(method='euclid', cutoff=17, sfreq=sfreq)
        raw_data = raw.get_data() 
        _, sample_mask = asr.fit(raw_data)
        raw._data = asr.transform(raw_data)
        
        ica = mne.preprocessing.ICA(n_components=19, method='infomax', fit_params=dict(extended=True), random_state=42)
        ica.fit(raw, verbose='ERROR')
        ic_labels = label_components(raw, ica, method='iclabel')
        ica.exclude = [idx for idx, label in enumerate(ic_labels['labels']) if label in ['eye blink', 'muscle artifact']]
        raw_cleaned = ica.apply(raw.copy(), verbose='ERROR')
        
        # Epoching (4s com 50% de overlap)[cite: 1]
        epochs = mne.make_fixed_length_epochs(raw_cleaned, duration=4.0, overlap=2.0, preload=True, verbose='ERROR')
        epochs_data = epochs.get_data()
        
        # Calcula exclusivamente a baseline (RBP) para cada época
        for epoch_idx, epoch_data in enumerate(epochs_data):
            epoch_id = f"{sub_id}_ep{epoch_idx:04d}"
            
            baseline_dict = compute_baseline_rbp(epoch_data, sfreq)
            baseline_dict['Epoch_ID'] = epoch_id
            
            baseline_features_list.append(baseline_dict)
            
    except Exception as e:
        print(f"[{sub_id}] Erro: {e}")

[sub-001] Extraindo Baseline (RBP)...


C:\Users\isabe\AppData\Local\Temp\ipykernel_33452\1738453476.py:15: RuntimeWarning: Location for this channel is unknown or ambiguous; consider calling set_montage() after adding new reference channels if needed. Applying a montage will only set locations of channels that exist at the time it is applied.
  raw = mne.add_reference_channels(raw, ref_channels=['A1', 'A2'])
C:\Users\isabe\AppData\Local\Temp\ipykernel_33452\1738453476.py:28: RuntimeWarning: The provided Raw instance is not filtered between 1 and 100 Hz. ICLabel was designed to classify features extracted from an EEG dataset bandpass filtered between 1 and 100 Hz (see the 'filter()' method for Raw and Epochs instances).
  ic_labels = label_components(raw, ica, method='iclabel')


[sub-002] Extraindo Baseline (RBP)...


C:\Users\isabe\AppData\Local\Temp\ipykernel_33452\1738453476.py:15: RuntimeWarning: Location for this channel is unknown or ambiguous; consider calling set_montage() after adding new reference channels if needed. Applying a montage will only set locations of channels that exist at the time it is applied.
  raw = mne.add_reference_channels(raw, ref_channels=['A1', 'A2'])
C:\Users\isabe\AppData\Local\Temp\ipykernel_33452\1738453476.py:28: RuntimeWarning: The provided Raw instance is not filtered between 1 and 100 Hz. ICLabel was designed to classify features extracted from an EEG dataset bandpass filtered between 1 and 100 Hz (see the 'filter()' method for Raw and Epochs instances).
  ic_labels = label_components(raw, ica, method='iclabel')


[sub-003] Extraindo Baseline (RBP)...


...

C:\Users\isabe\AppData\Local\Temp\ipykernel_33452\1738453476.py:15: RuntimeWarning: Location for this channel is unknown or ambiguous; consider calling set_montage() after adding new reference channels if needed. Applying a montage will only set locations of channels that exist at the time it is applied.
  raw = mne.add_reference_channels(raw, ref_channels=['A1', 'A2'])
C:\Users\isabe\AppData\Local\Temp\ipykernel_33452\1738453476.py:28: RuntimeWarning: The provided Raw instance is not filtered between 1 and 100 Hz. ICLabel was designed to classify features extracted from an EEG dataset bandpass filtered between 1 and 100 Hz (see the 'filter()' method for Raw and Epochs instances).
  ic_labels = label_components(raw, ica, method='iclabel')


[sub-064] Extraindo Baseline (RBP)...


C:\Users\isabe\AppData\Local\Temp\ipykernel_33452\1738453476.py:15: RuntimeWarning: Location for this channel is unknown or ambiguous; consider calling set_montage() after adding new reference channels if needed. Applying a montage will only set locations of channels that exist at the time it is applied.
  raw = mne.add_reference_channels(raw, ref_channels=['A1', 'A2'])
C:\Users\isabe\AppData\Local\Temp\ipykernel_33452\1738453476.py:28: RuntimeWarning: The provided Raw instance is not filtered between 1 and 100 Hz. ICLabel was designed to classify features extracted from an EEG dataset bandpass filtered between 1 and 100 Hz (see the 'filter()' method for Raw and Epochs instances).
  ic_labels = label_components(raw, ica, method='iclabel')


[sub-065] Extraindo Baseline (RBP)...


C:\Users\isabe\AppData\Local\Temp\ipykernel_33452\1738453476.py:15: RuntimeWarning: Location for this channel is unknown or ambiguous; consider calling set_montage() after adding new reference channels if needed. Applying a montage will only set locations of channels that exist at the time it is applied.
  raw = mne.add_reference_channels(raw, ref_channels=['A1', 'A2'])
C:\Users\isabe\AppData\Local\Temp\ipykernel_33452\1738453476.py:28: RuntimeWarning: The provided Raw instance is not filtered between 1 and 100 Hz. ICLabel was designed to classify features extracted from an EEG dataset bandpass filtered between 1 and 100 Hz (see the 'filter()' method for Raw and Epochs instances).
  ic_labels = label_components(raw, ica, method='iclabel')


In [20]:
# Transforma os resultados em um DataFrame
df_baseline = pd.DataFrame(baseline_features_list)

# Usa o 'Epoch_ID' para alinhar a baseline perfeitamente ao lado das features avançadas 
# (só mantém as épocas saudáveis que já existiam na tabela filtrada)
df_final_ml = pd.merge(df_filtered, df_baseline, on='Epoch_ID', how='inner')

# Salva o arquivo CSV definitivo para a modelagem univariada
df_final_ml.to_csv('eeg_features_baseline_data.csv', index=False)

print("\nConcluído!")
print(f"O dataset final tem {df_final_ml.shape[0]} amostras e {df_final_ml.shape[1]} colunas, contendo apenas pacientes de Alzheimer e Controles Saudáveis.")
display(df_final_ml.head())


Concluído!
O dataset final tem 26823 amostras e 34 colunas, contendo apenas pacientes de Alzheimer e Controles Saudáveis.


,Subject_ID,Epoch_ID,Delta_Mean_Freq,Theta_Mean_Freq,Alpha_Mean_Freq,Beta_Mean_Freq,Gamma_Mean_Freq,Delta_LZC,Theta_LZC,Alpha_LZC,...,MI_Theta,MI_Alpha,MI_Beta,MI_Gamma,Group,Delta_RBP,Theta_RBP,Alpha_RBP,Beta_RBP,Gamma_RBP
0,sub-001,sub-001_ep0000,1.640240,5.556964,9.578184,17.539659,32.905927,0.064929,0.157850,0.197673,...,0.782428,0.719417,0.557453,0.269773,A,0.871828,0.087822,0.030977,0.022074,0.014837
1,sub-001,sub-001_ep0001,1.263603,5.297219,9.978289,17.177341,32.587770,0.069258,0.145152,0.201713,...,0.655105,0.639406,0.589453,0.271156,A,0.858398,0.114329,0.024931,0.022349,0.009858
2,sub-001,sub-001_ep0002,1.241174,5.504575,10.279413,17.791362,32.251291,0.065506,0.138227,0.210081,...,0.726269,0.618728,0.592048,0.237782,A,0.876784,0.084128,0.019812,0.020459,0.010615
3,sub-001,sub-001_ep0003,1.896750,5.383784,10.495496,17.017366,31.464986,0.060889,0.145441,0.193056,...,0.839995,0.756799,0.673875,0.236825,A,0.806916,0.167413,0.040611,0.025160,0.014928
4,sub-001,sub-001_ep0004,1.362652,5.152837,10.775765,17.312952,31.387263,0.061755,0.153232,0.196518,...,0.683039,0.752469,0.672838,0.223810,A,0.861845,0.112876,0.035965,0.024684,0.010096
